Подключаюсь к спарку

In [8]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()
    

26/02/01 12:18:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Читаю таблицы

In [9]:
# ⬇️ Параметры подключения к PostgreSQL
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name = "public.regions"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

shop_timezone_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.shop_timezone")
    .option("user", db_user)
    .option("password", db_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)
shop_timezone_df.show(truncate=False)
shops_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.shops")
    .option("user", db_user)
    .option("password", db_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)
shops_df.show(truncate=False)


+-----+---------+
|plant|time_zone|
+-----+---------+
|842  |         |
|842  |RUS07    |
|843  |RUS04    |
|844  |         |
|845  |         |
|845  |RUS05    |
|847  |RUS03    |
|848  |RUS08    |
|848  |         |
|P847 |         |
|E103 |RUS08    |
|-134 |RUS04    |
|0    |         |
|0    |RUS08    |
|848  |         |
+-----+---------+

+-----+-----------+
|st_id|shop_name  |
+-----+-----------+
|842  |Lenta      |
|843  |Magnit     |
|844  |Spar       |
|845  |Pyaterochka|
|846  |Lenta      |
|847  |Diksi      |
|848  |Lenta      |
|849  |FixPrice   |
|850  |Magnit     |
|851  |Lenta      |
+-----+-----------+



Трансформация

In [ ]:
df = (
    shops_df.alias("s")
    .join(
        shop_timezone_df.alias("tz"),
        F.col("s.st_id") == F.col("tz.plant"),
        how="left"
    )
)

df_transformed = (
    df
    .withColumn(
        "tz_code",
        F.when(
            (F.col("tz.time_zone").isNull()) | (F.col("tz.time_zone") == ""),
            F.lit(3)
        )
        .when(F.col("tz.time_zone") == "RUS04", F.lit(4))
        .otherwise(
            F.regexp_extract(F.col("tz.time_zone"), r"RUS(\d+)", 1).cast("int")
        )
    )
    .select(
        F.col("s.st_id").alias("st_id"),
        F.col("s.shop_name"),
        F.col("tz_code")
    )
)

df_transformed.dropDuplicates(['st_id']).orderBy('st_id').show()



+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  846|      Lenta|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      3|
|  849|   FixPrice|      3|
|  850|     Magnit|      3|
|  851|      Lenta|      3|
+-----+-----------+-------+



остановка спарка

In [14]:

spark.stop()
